In [1]:
!pip install GitPython pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import pandas as pd

from git import Repo

In [3]:
repo_url = "https://github.com/psf/requests.git"

repo_name = repo_url.split("/")[-1].replace(".git", "")

print("Repository:", repo_name)

Repository: requests


In [6]:
import os

print(os.getcwd())

C:\Users\Tushti Gupta


In [7]:
import os

# Main project folder
project_dir = os.path.join(
    os.path.expanduser("~"),
    "AI_Software_Engineering_Agent"
)

# Required folders
raw_repo_dir = os.path.join(
    project_dir,
    "data",
    "raw_repositories"
)

extracted_dir = os.path.join(
    project_dir,
    "data",
    "extracted_code"
)

notebook_dir = os.path.join(
    project_dir,
    "notebooks"
)

# Create folders
os.makedirs(raw_repo_dir, exist_ok=True)
os.makedirs(extracted_dir, exist_ok=True)
os.makedirs(notebook_dir, exist_ok=True)

print("Project folders created successfully!")
print()
print("Project:", project_dir)
print("Raw repositories:", raw_repo_dir)
print("Extracted code:", extracted_dir)

Project folders created successfully!

Project: C:\Users\Tushti Gupta\AI_Software_Engineering_Agent
Raw repositories: C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\raw_repositories
Extracted code: C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code


In [8]:
print(os.path.exists(project_dir))
print(os.path.exists(raw_repo_dir))
print(os.path.exists(extracted_dir))

True
True
True


In [9]:
from git import Repo

repo_url = "https://github.com/psf/requests.git"

repo_name = repo_url.split("/")[-1].replace(".git", "")

repo_path = os.path.join(
    raw_repo_dir,
    repo_name
)

print("Repository will be stored at:")
print(repo_path)

Repository will be stored at:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\raw_repositories\requests


In [10]:
if not os.path.exists(repo_path):

    print("Downloading repository...")

    Repo.clone_from(
        repo_url,
        repo_path
    )

    print("Repository downloaded successfully!")

else:

    print("Repository already exists.")

Repository downloaded successfully!


In [11]:
print(os.listdir(repo_path)[:20])

['.coveragerc', '.git', '.git-blame-ignore-revs', '.github', '.gitignore', '.pre-commit-config.yaml', '.readthedocs.yaml', 'AUTHORS.rst', 'docs', 'ext', 'HISTORY.md', 'LICENSE', 'Makefile', 'MANIFEST.in', 'NOTICE', 'pyproject.toml', 'README.md', 'requirements-dev.txt', 'setup.py', 'src']


In [12]:
excluded_folders = {
    ".git",
    "__pycache__",
    "venv",
    ".venv",
    "env",
    "node_modules",
    "build",
    "dist",
    ".idea",
    ".vscode"
}

dataset = []

for root, dirs, files in os.walk(repo_path):

    # Exclude unnecessary directories
    dirs[:] = [
        d for d in dirs
        if d not in excluded_folders
    ]

    for file in files:

        # Only extract Python files
        if not file.endswith(".py"):
            continue

        file_path = os.path.join(root, file)

        try:

            with open(
                file_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                code = f.read()

            # Skip empty files
            if not code.strip():
                continue

            relative_path = os.path.relpath(
                file_path,
                repo_path
            )

            record = {
                "repository": repo_name,
                "repository_url": repo_url,
                "file_name": file,
                "file_path": relative_path,
                "language": "Python",
                "file_size_bytes": os.path.getsize(file_path),
                "code": code
            }

            dataset.append(record)

        except Exception as e:

            print(
                f"Could not read {file_path}: {e}"
            )

print("Extraction completed!")
print("Python files extracted:", len(dataset))

Extraction completed!
Python files extracted: 36


In [13]:
json_path = os.path.join(
    extracted_dir,
    "code_dataset.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset,
        f,
        indent=4,
        ensure_ascii=False
    )

print("JSON saved:")
print(json_path)

JSON saved:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_dataset.json


In [14]:
metadata = []

for record in dataset:

    metadata.append({
        "repository": record["repository"],
        "repository_url": record["repository_url"],
        "file_name": record["file_name"],
        "file_path": record["file_path"],
        "language": record["language"],
        "file_size_bytes": record["file_size_bytes"]
    })

df = pd.DataFrame(metadata)

csv_path = os.path.join(
    extracted_dir,
    "code_metadata.csv"
)

df.to_csv(
    csv_path,
    index=False
)

print("CSV saved:")
print(csv_path)

CSV saved:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_metadata.csv


In [15]:
print("Total Python files extracted:", len(dataset))

Total Python files extracted: 36


In [16]:
print("JSON file exists:", os.path.exists(json_path))
print("JSON path:", json_path)

JSON file exists: True
JSON path: C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_dataset.json


In [17]:
print("First file:")
print(dataset[0]["file_name"])

print("\nFile path:")
print(dataset[0]["file_path"])

print("\nCode preview:")
print(dataset[0]["code"][:500])

First file:
setup.py

File path:
setup.py

Code preview:
import sys

if sys.version_info < (3, 10):  # noqa: UP036
    sys.stderr.write("Requests requires Python 3.10 or later.\n")
    sys.exit(1)

from setuptools import setup

setup()



In [18]:
df.head()

,repository,repository_url,file_name,file_path,language,file_size_bytes
0,requests,https://github.com/psf/requests.git,setup.py,setup.py,Python,188
1,requests,https://github.com/psf/requests.git,conf.py,docs\conf.py,Python,12533
2,requests,https://github.com/psf/requests.git,flask_theme_support.py,docs\_themes\flask_theme_support.py,Python,4961
3,requests,https://github.com/psf/requests.git,adapters.py,src\requests\adapters.py,Python,28740
4,requests,https://github.com/psf/requests.git,api.py,src\requests\api.py,Python,7332


In [19]:
print("Total Python files extracted:", len(dataset))

Total Python files extracted: 36


In [20]:
print(dataset[0]["code"][:500])

import sys

if sys.version_info < (3, 10):  # noqa: UP036
    sys.stderr.write("Requests requires Python 3.10 or later.\n")
    sys.exit(1)

from setuptools import setup

setup()



In [21]:
## MULTIPLE REPOSITORIES EXTRACTION

In [22]:
import os
import json
import pandas as pd

from git import Repo

project_dir = os.path.join(
    os.path.expanduser("~"),
    "AI_Software_Engineering_Agent"
)

raw_repo_dir = os.path.join(
    project_dir,
    "data",
    "raw_repositories"
)

extracted_dir = os.path.join(
    project_dir,
    "data",
    "extracted_code"
)

os.makedirs(raw_repo_dir, exist_ok=True)
os.makedirs(extracted_dir, exist_ok=True)

print("Project directory:")
print(project_dir)

Project directory:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent


In [23]:
repositories = [

    {
        "name": "requests",
        "url": "https://github.com/psf/requests.git"
    },

    {
        "name": "flask",
        "url": "https://github.com/pallets/flask.git"
    },

    {
        "name": "fastapi",
        "url": "https://github.com/tiangolo/fastapi.git"
    },

    {
        "name": "pydantic",
        "url": "https://github.com/pydantic/pydantic.git"
    },

    {
        "name": "httpx",
        "url": "https://github.com/encode/httpx.git"
    },

    {
        "name": "pytest",
        "url": "https://github.com/pytest-dev/pytest.git"
    },

    {
        "name": "black",
        "url": "https://github.com/psf/black.git"
    },

    {
        "name": "rich",
        "url": "https://github.com/Textualize/rich.git"
    },

    {
        "name": "click",
        "url": "https://github.com/pallets/click.git"
    },

    {
        "name": "typer",
        "url": "https://github.com/fastapi/typer.git"
    }
]

print("Total repositories:", len(repositories))

Total repositories: 10


In [24]:
excluded_folders = {
    ".git",
    "__pycache__",
    "venv",
    ".venv",
    "env",
    "node_modules",
    "build",
    "dist",
    ".idea",
    ".vscode",
    ".tox",
    ".pytest_cache",
    ".mypy_cache",
    ".ruff_cache",
    ".github"
}

In [25]:
for repo_info in repositories:

    repo_name = repo_info["name"]
    repo_url = repo_info["url"]

    repo_path = os.path.join(
        raw_repo_dir,
        repo_name
    )

    print("\n" + "=" * 60)
    print("Repository:", repo_name)
    print("=" * 60)

    if os.path.exists(repo_path):

        print("Already exists. Skipping download.")

    else:

        try:

            print("Downloading...")

            Repo.clone_from(
                repo_url,
                repo_path
            )

            print("Downloaded successfully.")

        except Exception as e:

            print("ERROR:", e)


Repository: requests
Already exists. Skipping download.

Repository: flask
Downloading...
Downloaded successfully.

Repository: fastapi
Downloading...
Downloaded successfully.

Repository: pydantic
Downloading...
Downloaded successfully.

Repository: httpx
Downloading...
Downloaded successfully.

Repository: pytest
Downloading...
Downloaded successfully.

Repository: black
Downloading...
Downloaded successfully.

Repository: rich
Downloading...
Downloaded successfully.

Repository: click
Downloading...
Downloaded successfully.

Repository: typer
Downloading...
Downloaded successfully.


In [26]:
dataset = []

record_id = 0

for repo_info in repositories:

    repo_name = repo_info["name"]
    repo_url = repo_info["url"]

    repo_path = os.path.join(
        raw_repo_dir,
        repo_name
    )

    print("\nProcessing:", repo_name)

    repo_file_count = 0

    for root, dirs, files in os.walk(repo_path):

        # Exclude unnecessary directories
        dirs[:] = [
            d for d in dirs
            if d not in excluded_folders
        ]

        for file in files:

            # Only Python source files
            if not file.endswith(".py"):
                continue

            file_path = os.path.join(
                root,
                file
            )

            try:

                with open(
                    file_path,
                    "r",
                    encoding="utf-8",
                    errors="ignore"
                ) as f:

                    code = f.read()

                # Skip empty files
                if not code.strip():
                    continue

                relative_path = os.path.relpath(
                    file_path,
                    repo_path
                )

                record_id += 1
                repo_file_count += 1

                record = {

                    "id": record_id,

                    "repository": repo_name,

                    "repository_url": repo_url,

                    "file_name": file,

                    "file_path": relative_path,

                    "language": "Python",

                    "file_size_bytes":
                        os.path.getsize(file_path),

                    "code": code
                }

                dataset.append(record)

            except Exception as e:

                print(
                    f"Could not read {file_path}: {e}"
                )

    print(
        "Python files extracted:",
        repo_file_count
    )

print("\n" + "=" * 60)
print("TOTAL EXTRACTION COMPLETE")
print("=" * 60)

print(
    "Total Python files:",
    len(dataset)
)


Processing: requests
Python files extracted: 36

Processing: flask
Python files extracted: 80

Processing: fastapi
Python files extracted: 955

Processing: pydantic
Python files extracted: 429

Processing: httpx
Python files extracted: 57

Processing: pytest
Python files extracted: 265

Processing: black
Python files extracted: 321

Processing: rich
Python files extracted: 211

Processing: click
Python files extracted: 85

Processing: typer
Python files extracted: 521

TOTAL EXTRACTION COMPLETE
Total Python files: 2960


In [27]:
repo_counts = {}

for record in dataset:

    repo = record["repository"]

    repo_counts[repo] = (
        repo_counts.get(repo, 0) + 1
    )

repo_counts

{'requests': 36,
 'flask': 80,
 'fastapi': 955,
 'pydantic': 429,
 'httpx': 57,
 'pytest': 265,
 'black': 321,
 'rich': 211,
 'click': 85,
 'typer': 521}

In [28]:
repo_distribution = pd.DataFrame(
    list(repo_counts.items()),
    columns=[
        "repository",
        "python_files"
    ]
)

repo_distribution

,repository,python_files
0,requests,36
1,flask,80
2,fastapi,955
3,pydantic,429
4,httpx,57
5,pytest,265
6,black,321
7,rich,211
8,click,85
9,typer,521


In [29]:
json_path = os.path.join(
    extracted_dir,
    "code_dataset.json"
)

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        dataset,
        f,
        indent=4,
        ensure_ascii=False
    )

print("JSON dataset saved:")
print(json_path)

JSON dataset saved:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_dataset.json


In [30]:
metadata = []

for record in dataset:

    metadata.append({

        "id": record["id"],

        "repository":
            record["repository"],

        "repository_url":
            record["repository_url"],

        "file_name":
            record["file_name"],

        "file_path":
            record["file_path"],

        "language":
            record["language"],

        "file_size_bytes":
            record["file_size_bytes"]
    })

df = pd.DataFrame(metadata)

csv_path = os.path.join(
    extracted_dir,
    "code_metadata.csv"
)

df.to_csv(
    csv_path,
    index=False
)

print("CSV dataset saved:")
print(csv_path)

CSV dataset saved:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_metadata.csv


In [31]:
print("=" * 60)
print("PHASE 1 FINAL DATASET")
print("=" * 60)

print("Repositories:", len(repositories))
print("Total Python files:", len(dataset))
print("Total metadata records:", len(df))

print("\nRepository distribution:")
print(repo_distribution)

print("\nJSON:")
print(json_path)

print("\nCSV:")
print(csv_path)

PHASE 1 FINAL DATASET
Repositories: 10
Total Python files: 2960
Total metadata records: 2960

Repository distribution:
  repository  python_files
0   requests            36
1      flask            80
2    fastapi           955
3   pydantic           429
4      httpx            57
5     pytest           265
6      black           321
7       rich           211
8      click            85
9      typer           521

JSON:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_dataset.json

CSV:
C:\Users\Tushti Gupta\AI_Software_Engineering_Agent\data\extracted_code\code_metadata.csv


In [32]:
print("Repository:")
print(dataset[0]["repository"])

print("\nFile:")
print(dataset[0]["file_name"])

print("\nPath:")
print(dataset[0]["file_path"])

print("\nCode:")
print(dataset[0]["code"][:1000])

Repository:
requests

File:
setup.py

Path:
setup.py

Code:
import sys

if sys.version_info < (3, 10):  # noqa: UP036
    sys.stderr.write("Requests requires Python 3.10 or later.\n")
    sys.exit(1)

from setuptools import setup

setup()



In [33]:
print("Total Python files:", len(dataset))
print(repo_distribution)

Total Python files: 2960
  repository  python_files
0   requests            36
1      flask            80
2    fastapi           955
3   pydantic           429
4      httpx            57
5     pytest           265
6      black           321
7       rich           211
8      click            85
9      typer           521
